# M1 · Detección de placas + OCR (validación)
**IAM-811 · Laboratorio 3**

Se reutiliza el detector del proyecto anterior (`yolo-v9-t-384-license-plates-end2end.onnx`) y se mide la precisión del OCR
(fast-plate-ocr como motor principal y EasyOCR como comparación).

Subir a Colab una carpeta `placas_eval/` con fotos y un `referencias.txt` con líneas `01.jpg=1234ABC`
(por ejemplo, las 20 fotos `PLACAS_H1900-2130_8405213/INFERENCIA` con sus placas reales, más fotos propias de placas bolivianas).

In [ ]:
!pip install -q onnxruntime==1.20.1 fast-plate-ocr==1.1.0 easyocr==1.7.2 opencv-python-headless==4.10.0.84 fastapi==0.115.6 python-multipart==0.0.20 ultralytics==8.3.240
!git clone -q https://github.com/TU_USUARIO/peaje-ia.git repo   # <-- cambiar por tu repositorio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/peaje_evidencias'
import os; os.makedirs(SALIDA, exist_ok=True)

In [ ]:
import sys, os, re, time, cv2, numpy as np
sys.path.insert(0, 'repo/inferencia')
os.environ['DISPOSITIVO'] = 'cpu'
import app as inf   # mismo código del endpoint

CARPETA = '/content/placas_eval'   # <-- subir aquí fotos + referencias.txt
refs = dict(l.strip().split('=') for l in open(f'{CARPETA}/referencias.txt') if '=' in l and l.strip().split('=')[1])

def cer(a, b):
    d = np.arange(len(b) + 1)
    for i, ca in enumerate(a, 1):
        prev, d[0] = d[0], i
        for j, cb in enumerate(b, 1):
            prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (ca != cb))
    return d[len(b)] / max(1, len(b))

for motor in ['fastocr', 'easyocr']:
    inf.OCR_MOTOR = motor
    aciertos, cers, detectadas, tiempos = 0, [], 0, []
    for nombre, real in sorted(refs.items()):
        img = cv2.imread(f'{CARPETA}/{nombre}')
        t = time.perf_counter()
        cajas = inf.detectar_placas(img)
        texto = inf.normalizar(inf.leer_placa(img, cajas[0])[0]) if cajas else ''
        tiempos.append((time.perf_counter() - t) * 1000)
        detectadas += bool(cajas)
        aciertos += texto == inf.normalizar(real)
        cers.append(cer(texto, inf.normalizar(real)))
        print(f'{motor:8s} {nombre}: real={real:10s} leído={texto:10s} {"✓" if texto == inf.normalizar(real) else "✗"}')
    n = len(refs)
    print(f'\n== {motor}: detección {detectadas}/{n} | exactas {aciertos}/{n} = {aciertos/n:.1%} | CER {np.mean(cers):.3f} | {np.mean(tiempos):.0f} ms/imagen\n')